NameError: name 'eps' is not defined

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

In [7]:
import numpy as np
import pandas as pd
import yfinance as yf
import time

# 1. Define the Master List
master_aim_list = [
    "ABDP.L", "AMS.L", "AOM.L", "ANCR.L", "AVG.L", "BPM.L", "BKS.L", "BIG.L", 
    "BOKU.L", "BRCK.L", "BUR.L", "CAML.L", "CER.L", "CHRT.L", "CNC.L", "CRW.L", 
    "DOTD.L", "EVPL.L", "FNTL.L", "FNX.L", "FRAN.L", "FRP.L", "GHH.L", "HSP.L", 
    "HUW.L", "JHD.L", "JET2.L", "KEYS.L", "KGH.L", "LTHM.L", "MPE.L", "SAA.L", 
    "MHA.L", "MIDW.L", "NICL.L", "PHP.L", "POLR.L", "RWS.L", "SRC.L", "SAVE.L",
    "TFW.L", "VIC.L", "VCP.L", "WJG.L", "YOU.L", "GBG.L", "DATA.L", "IPX.L"
]

# Set the active tickers to the master list
tickers = master_aim_list
results = []

print(f"Starting scan of {len(tickers)} AIM stocks...")

# 2. The Engine
for t in tickers:
    try:
        s = yf.Ticker(t)
        info = s.info
        
        # Get raw values (default to 0 if missing)
        raw_price = info.get('currentPrice', 0)
        eps = info.get('trailingEps', 0)
        bvps = info.get('bookValue', 0)
        name = info.get('longName', t)
        
        # Fix the "Pence vs Pounds" glitch
        # UK stocks over 10 are usually in pence (e.g., 1190 = £11.90)
        price = raw_price / 100 if raw_price > 10 else raw_price
        
        # 3. Calculation: Only if profitable (cannot sqrt a negative)
        if eps and bvps and eps > 0 and bvps > 0:
            intrinsic = np.sqrt(22.5 * eps * bvps)
            margin = (intrinsic - price) / intrinsic
        else:
            intrinsic = 0
            margin = -1 # Represents -100%
            
        results.append({
            "Ticker": t,
            "Name": name[:20],
            "Price (£)": round(price, 2),
            "Intrinsic (£)": round(intrinsic, 2),
            "Margin %": margin # Kept as float for sorting
        })
        
        # Small sleep to be polite to Yahoo Finance servers
        time.sleep(0.1)
        
    except Exception as e:
        print(f"Skipping {t}: Data unavailable")

# 4. Create DataFrame and Format Output
df = pd.DataFrame(results)

# Sort by Margin % so the biggest bargains are at the top
df = df.sort_values(by="Margin %", ascending=False)

# Format the Margin % column for display
df["Margin %"] = df["Margin %"].apply(lambda x: f"{x:.1%}")

print("Scan Complete!")
df

Starting scan of 48 AIM stocks...
Scan Complete!


,Ticker,Name,Price (£),Intrinsic (£),Margin %
5,BPM.L,B.P. Marsh & Partner,6.90,16.02,56.9%
26,JET2.L,Jet2 plc,11.93,23.31,48.8%
24,HUW.L,Helios Underwriting,2.18,4.07,46.5%
29,LTHM.L,James Latham plc,10.00,14.88,32.8%
47,IPX.L,Impax Asset Manageme,1.00,1.44,30.9%
35,PHP.L,Primary Health Prope,0.94,1.34,29.8%
9,BRCK.L,BRCK Group plc,0.48,0.63,23.6%
23,HSP.L,Hargreaves Services,7.94,9.45,16.0%
30,MPE.L,M.P. Evans Group PLC,15.38,17.38,11.5%
17,EVPL.L,everplay group plc,2.78,2.94,5.4%


In [8]:
# This saves your table to a CSV file in your project folder
df.to_csv("AIM_Intrinsic_Value_Report_May_2026.csv", index=False)
print("File Saved Successfully!")


File Saved Successfully!


In [4]:
# Assuming 'df' is the name of your results table
# We filter for positive intrinsic value and sort by the best bargain
true_bargains = df[df['Intrinsic (£)'] > 0].sort_values(by="Margin %", ascending=False)

print("--- PROFITABLE AIM BARGAINS ---")
true_bargains

--- PROFITABLE AIM BARGAINS ---


,Ticker,Price (£),Intrinsic (£),Margin %
3,JET2.L,11.90,23.31,48.9%
1,GGP.L,7.31,5.36,-36.4%


In [ ]:
# 1. Pick a stock to test
ticker_symbol = "BOO.L" # Boohoo Group (AIM Penny Stock)
stock = yf.Ticker(ticker_symbol)
info = stock.info

# 2. Define the variables from the data
eps = info.get('trailingEps')
book_value = info.get('bookValue')
price = info.get('currentPrice')

# 3. NOW run the calculation
if eps and book_value and eps > 0:
    intrinsic_value = np.sqrt(22.5 * eps * book_value)
    margin = (intrinsic_value - price) / intrinsic_value
    print(f"Analysis for {ticker_symbol}:")
    print(f"Intrinsic Value: £{intrinsic_value:.2f}")
    print(f"Current Price:   £{price:.2f}")
    print(f"Margin of Safety: {margin:.1%}")
else:
    print(f"Could not calculate for {ticker_symbol}. Check if EPS is negative.")

Markdown
### Graham Number Valuation
We use the Graham Number to estimate the maximum fair price:
$$Intrinsic Value = \sqrt{22.5 \times \text{EPS} \times \text{Book Value per Share}}$$